# <center>The Dantzig–Wolfe decomposition</center>
### <center>Alfred Galichon (NYU & Sciences Po), Antoine Jacquet (Sciences Po) and Georgy Salakhutdinov (Polytechnique)</center>
## <center>'math+econ+code' masterclass series, Collegio Carlo Alberto, March 2026</center>
#### <center>With python code examples</center>
© 2018–2026 by Alfred Galichon. Past and present support from NSF grant DMS-1716489, ERC grant CoG-866274 are acknowledged, as well as inputs from contributors listed [here](https://www.google.com/url?q=https://www.google.com/url?q%3Dhttp://www.math-econ-code.org/team).

**If you reuse material from this masterclass, please cite as:**<br>
Alfred Galichon, 'math+econ+code' masterclass series. https://www.math-econ-code.org/

## Learning objectives

* Large-scale linear programming
* Dantzig—Wolfe decomposition
* Column generation


## References

* Dantzig & Wolfe (1960). Decomposition Principle for Linear Programs. *Operations Research*
* Galichon, Jacquet, Salakhutdinov (2025). Transferable Utility Matching Beyond Logit: Computation and Estimation with General Heterogeneity. *Working paper*, https://arxiv.org/abs/2511.23116

## Libraries


In [8]:
import numpy as np
import gurobipy as grb

## Introduction

When a linear program is too large to solve efficiently as a single model, it can be better to exploit its structure.
The *Dantzig–Wolfe decomposition* does this for LPs with a suitable *block-diagonal form*: it splits the problem into several *independent subproblems* and a coordinating *master problem*.
The solving algorithm then alternates between them.

The key benefit is scalability: the method starts with a small set of variables and adds more only when they improve the solution, instead of optimizing over all variables from the start.


## Motivation and setting

We will consider the following problem as motivation.
A profit-maximizing planner allocates production across $K$ factories $k=1,\dots,K$.
Each factory can produce multiple products (say, different car models), and the set of products may even be different across factories.
However, each factory operates under both local constraints and global constraints on its inputs.
Local inputs are for instance work hours or machine time.
Global inputs are for instance the different materials required to build the cars, that can be allocated to the different factories by the planner.

**Decisions.**
Let $G_k$ be the set of cars that can be produced by factory $k$.  
The planner chooses for each factory $k$ the quantity $x_g^k$ of model-$g$ cars to produce for all $g \in G_k$,
\begin{equation}
x^k = (x_g^k)_{g \in G_k} \geq 0.
\end{equation}

**Profit.**
Each car $g \in G_k$ produced by factory $k$ is sold for profit $c_g^k$.  
The planner wants to maximize the total profit,
\begin{equation}
\sum_{k=1}^K \sum_{g \in G_k} c_g^k \, x_g^k = \sum_{k=1}^K (c^k)^\top x^k.
\end{equation}

**Local constraints on inputs.**
In each factory $k$, production uses a collection of local inputs $i \in I_k$ (labor-hours, components, machine time, etc).
Producing one car $g$ in factory $k$ requires $a_{ig}^k$ units of input $i$ (different factories may have different production technologies).
The available quantity of local input $i$ in factory $k$ is $b_i^k$.  
Thus, for each input $i \in I_k$ and factory $k$,
\begin{equation}
\sum_{g \in G_t} a_{ig}^k \, x_g^k \leq b_i^k,
\qquad \text{or in matrix form,} \qquad
A^k \, x^k \leq b^k
\end{equation}
where $A^k = (a_{ig}^k)$ is an $I_k \times G_k$ matrix.

**Global constraints on inputs.**
The planner must allocate global resources across the different factories (e.g., raw materials needed to build cars) which are indexed by $j \in J_0$.
Again because factories may have different production technologies, producing one model-$g$ car in factory $k$ consumes $d_{jg}^k$ of input $j$. 
The total available amount of input $j$ to be allocated is $b_j^0$.
Thus for each global input $j \in J_0$,
\begin{equation}
\sum_{k=1}^K \sum_{g \in G_t} d_{jg}^k \, x_g^k \leq b_j^0,
\qquad \text{or in matrix form,} \qquad
\sum_{k=1}^K D^k \, x^k \leq b^0
\end{equation}
where $D^k = (d_{jg}^k)$ is a $J_0 \times G_k$ matrix.


### LP formulation

It is clear that the above problem is a linear program.
However, because some constraints are only within-period while others are across-period, it admits a special structure.

To see this, stack all decisions into one long vector $x = (x^1,\dots,x^K) \geq 0$, $c = (c^1,\dots,c^K)$.
Then the planning problem is the LP
\begin{align}
\max_{x \geq 0} ~ & c^\top x \\
\text{s.t.} ~ & A x \leq b
\end{align}
where
\begin{equation}
A = 
\begin{pmatrix}
A_1 & 0   & \cdots & 0 \\
0   & A_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots\\
0   & 0   & \cdots & A_K \\
D_1 & D_2 & \cdots & D_K
\end{pmatrix}
\qquad \text{and} \qquad
b = 
\begin{pmatrix}
b_1 \\ b_2 \\ \vdots\\ b_K \\ b_0
\end{pmatrix}.
\end{equation}

- The **block-diagonal** constraints $A^k x^k \leq b^k$ encode *local input constraints*.
- The **linking** constraints $\sum_{k=1}^K D^k x^k \leq b^0$ encode *global input constraints*.

This is the canonical structure for Dantzig–Wolfe: many local blocks plus a small set of global linking constraints.

We encode this block structure in Python in a `DantzigWolfe` Python class as follows:
- `A_k`: list of subproblem constraint matrices $\left[A^1, \ldots, A^K\right]$, where $A^k x^k \leq b^k \quad \forall k \in [1, \dots, K]$
- `b_k`: list of RHS constraints vectors $\left[b^1, \ldots, b^K, b^0\right]$, the first $K$ entries are the subproblem constraints, and the last entry $b^0$ is the linking constraint
- `D_k`: list of linking constraint matrices $\left[D^1, \ldots, D^K\right]$, where $\sum_k D^k x^k \leq b^0$
- `c_k`: list of objective vectors $\left[c^1, \ldots, c^K\right]$, where the objective is $\sum_k (c^k)^\top x^k$

In [9]:
class DantzigWolfe():
    
    def __init__(self, A_k, D_k, b_k, c_k):
        self.K = len(A_k)
        self.G_k = [ A_k[k].shape[1] for k in range(self.K) ]
        if len(D_k) != self.K or len(b_k) != self.K+1 or len(c_k) != self.K:
            raise ValueError(f"Dimensions of inputs do not match.")
        self.A_k, self.D_k, self.b_k, self.c_k = A_k, D_k, b_k, c_k
        A_block = np.block([ [ A_k[i] if i == j else np.zeros((A_k[i].shape[0], A_k[j].shape[1]))
                               for j in range(self.K) ]
                             for i in range(self.K) ])
        self.A = np.vstack((A_block,
                       np.hstack(D_k)))
        self.b, self.c = np.hstack(b_k), np.hstack(c_k)

In [10]:
# Example: 5 periods, 3-4 car models each, 3 inputs, 4 emission types

A_k = [
    # t=0: 4 models, constrained by labour, components, machine time
    np.array([[2., 1., 3., 1.],
              [4., 3., 2., 5.],
              [1., 2., 4., 1.]]),
    # t=1: 3 models
    np.array([[3., 2., 4.],
              [1., 5., 2.],
              [2., 1., 3.]]),
    # t=2: 4 models
    np.array([[1., 3., 2., 2.],
              [3., 1., 4., 1.],
              [2., 2., 1., 3.]]),
    # t=3: 3 models
    np.array([[4., 1., 2.],
              [2., 3., 1.],
              [1., 2., 5.]]),
    # t=4: 4 models
    np.array([[2., 2., 1., 3.],
              [1., 3., 4., 2.],
              [3., 1., 2., 1.]]),
]

D_k = [
    # t=0: emissions per car (CO2, waste, NOx, particulates)
    np.array([[3., 2., 5., 1.],
              [1., 3., 1., 2.],
              [0., 1., 2., 0.],
              [2., 2., 3., 1.]]),
    # t=1
    np.array([[2., 1., 3.],
              [4., 2., 1.],
              [3., 0., 2.],
              [1., 3., 2.]]),
    # t=2
    np.array([[1., 4., 2., 1.],
              [2., 1., 3., 2.],
              [1., 0., 1., 3.],
              [3., 2., 1., 2.]]),
    # t=3
    np.array([[4., 1., 2.],
              [1., 3., 2.],
              [2., 1., 0.],
              [1., 2., 3.]]),
    # t=4
    np.array([[2., 3., 1., 2.],
              [1., 2., 4., 1.],
              [0., 2., 1., 1.],
              [2., 1., 3., 2.]]),
]

b_k = [
    np.array([120., 200., 150.]),        # t=0 input capacities
    np.array([90., 110., 80.]),          # t=1
    np.array([140., 160., 130.]),        # t=2
    np.array([100., 130., 110.]),        # t=3
    np.array([130., 150., 120.]),        # t=4
    np.array([500., 300., 120., 800.]),  # emission quotas over all periods
]

c_k = [
    np.array([12., 9., 15., 7.]),    # t=0 profit per car
    np.array([20., 14., 18.]),       # t=1
    np.array([8., 13., 10., 11.]),   # t=2
    np.array([14., 11., 9.]),        # t=3
    np.array([10., 16., 8., 12.]),   # t=4
]

ex = DantzigWolfe(A_k, D_k, b_k, c_k)

In [11]:
def DantzigWolfe_gurobi_solve(self, verbose=0):
    m = grb.Model()
    x = m.addMVar(np.sum(self.G_k), lb=0.0)
    m.setObjective(self.c.T @ x, grb.GRB.MAXIMIZE)
    m.addConstr(self.A @ x <= self.b)
    m.optimize()
    return m.ObjVal, m.X

DantzigWolfe.gurobi_solve = DantzigWolfe_gurobi_solve

In [12]:
ex.gurobi_solve()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 19 rows, 18 columns and 120 nonzeros
Model fingerprint: 0x67060b54
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [7e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 8e+02]
Presolve time: 0.01s
Presolved: 19 rows, 18 columns, 120 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2900000e+32   5.737500e+31   2.290000e+02      0s
      24    2.5932795e+03   0.000000e+00   0.000000e+00      0s

Solved in 24 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.593279533e+03


(2593.279532967033,
 [50.0,
  0.0,
  0.0,
  0.0,
  0.0,
  16.25,
  14.375,
  0.0,
  34.87980769230769,
  0.0,
  13.643543956043954,
  0.0,
  28.890796703296704,
  0.0,
  32.857142857142854,
  0.0,
  0.0,
  21.42857142857143])

## Dantzig–Wolfe decomposition

The Dantzig—Wolfe decomposition is a method to solve LPs of the above form at very large scales.
To do this, it reduces the number of variables of the problem and iteratively adds new ones if this is needed.

### Master problem formulation

For each $k$, consider the set of non-zero extreme points $\{x^{k,p}\}_{p \in P_k}$ (where $P_k$ is an enumerating index set) of the polytope defined by the subproblem:
\begin{equation}
A^k x^k \leq b^k \quad \text{with} \quad x^k \geq 0.
\end{equation}

Our previous linear program can then be recast as one where we choose how much weight $\lambda^{k,p}$ to put on each extreme point $p$ of subproblem $k$. Namely:

\begin{align}
\max_{\lambda^{k,p} \geq 0} ~ & \sum_{k=1}^K \sum_{p \in P_k} \lambda^{k,p} \big( (c^k)^\top x^{k,p} \big) \tag{M} \\
\text{s.t.} ~ & \sum_{k=1}^K \sum_{p\in P_k} \lambda^{k,p} (D^k \, x^{k,p}) \leq b^0 \\
              & \sum_{p \in P_k} \lambda^{k,p} \leq 1 \quad (\forall k).
\end{align}

We call this the **full master problem**.

Initially, we know that the vector $0_{G_k}$ is an extreme point for subproblem $k$, but we do not know what the other extreme points are.
Still, we code a function which initializes the master problem while keeping all $\lambda^{k,p}$ equal to 0 (that is, without any nontrivial extreme points).


In [13]:
def DantzigWolfe_initialize_master(self):
    m = grb.Model()
    m.Params.OutputFlag = 0
    m.setObjective(0, grb.GRB.MAXIMIZE)
    self.linking_j = [ m.addLConstr(0.0, grb.GRB.LESS_EQUAL, self.b_k[-1][j]) for j in range(len(self.b_k[-1])) ] # initialize 0 <= b^0
    self.convexity_k = [ m.addLConstr(0.0, grb.GRB.LESS_EQUAL, 1.0) for k in range(self.K) ]                      # initialize 0 <= 1
    self.lambda_vars = {}
    self.master = m

DantzigWolfe.initialize_master = DantzigWolfe_initialize_master

In [14]:
ex.initialize_master()

To find a first nontrivial extreme point for each subproblem $k$, we can simply solve that subproblem:
\begin{align}
\max_{x^k \geq 0} ~ & (c^k)^\top x^k \\
\text{s.t.} ~ & A^k x^k \leq b^k.
\end{align}

We find these first points with `initalize_extreme_points` and add the corresponding variable symbols $\lambda^{k,p}$, `self.lambda_vars[k,self.p_k[k]]`, to our master problem with `update_master`.
Note that this is still simply setting up the problem, not numerically solving for $\lambda^{k,p}$ yet.

This corresponds to activating a new column (objective coefficient and constraint matrix) in our problem (i.e. using a new variable), hence the name **column generation**.


In [15]:
def DantzigWolfe_initalize_extreme_points(self):
    self.x_k_p = [ [] for _ in range(self.K) ]
    self.p_k = [ 0 for _ in range(self.K) ]
    new_x_k = [ False for _ in range(self.K) ]
    for k in range(self.K):
        m = grb.Model()
        x = m.addMVar(self.G_k[k], lb=0.0)
        m.setObjective(self.c_k[k].T @ x, grb.GRB.MAXIMIZE)
        m.addConstr(self.A_k[k] @ x <= self.b_k[k])
        m.Params.OutputFlag = 0
        m.optimize()
        new_x_k[k] = m.X
        self.x_k_p[k].append(m.X)
    return new_x_k

DantzigWolfe.initalize_extreme_points = DantzigWolfe_initalize_extreme_points


def DantzigWolfe_update_master(self, new_x_k):
    for k in range(self.K):
        if new_x_k[k]:
            col = grb.Column()
            col.addTerms((self.D_k[k] @ new_x_k[k]).tolist(), self.linking_j)
            col.addTerms(1.0, self.convexity_k[k])
            self.lambda_vars[k,self.p_k[k]] = self.master.addVar( obj=float(self.c_k[k] @ new_x_k[k]), lb=0.0, column=col)

DantzigWolfe.update_master = DantzigWolfe_update_master


In [16]:
new_x_k = ex.initalize_extreme_points()
ex.update_master(new_x_k)

Now that our problem is updated with new extreme points, we can consider the restricted version of the master problem which only uses these points.
We call this the **restricted master problem**, and we solve it.

In [17]:
ex.master.Params.OutputFlag = 1
ex.master.optimize()
ex.master.ObjVal

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 9 rows, 5 columns and 25 nonzeros
Model fingerprint: 0x5d607ad9
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e+02, 9e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]
Presolve removed 6 rows and 0 columns
Presolve time: 0.00s
Presolved: 3 rows, 5 columns, 15 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.6857143e+03   2.666786e+02   0.000000e+00      0s
       3    1.6524038e+03   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.652403846e+03


1652.4038461538462

The value of the objective is around `1652`, which is lower than the optimal one, `2593`.
This is normal: we haven't considered the full set of extreme points yet.

### Column generation

Let's first derive the dual of the full master problem:

\begin{align}
& \max_{\lambda^{k,p} \geq 0} ~ \bigg\{ \sum_{k=1}^K \sum_{p \in P_k} \lambda^{k,p} \big( (c^k)^\top x^{k,p} \big)
\quad \text{s.t.} ~ \sum_{k=1}^K \sum_{p\in P_k} \lambda^{k,p} (D^k \, x^{k,p}) \leq b^0,
\quad \sum_{p \in P_k} \lambda^{k,p} \leq 1 \quad (\forall k) \bigg\} \\
= & \max_{\lambda^{k,p} \geq 0} ~ \min_{\sigma^k, \pi_j \geq 0} ~
\sum_{k=1}^K \sum_{p \in P_k} \lambda^{k,p} \big( (c^k)^\top x^{k,p} \big)
+ \pi^\top (b^0 - \sum_{k=1}^K \sum_{p\in P_k} \lambda^{k,p} (D^k \, x^{k,p}))
+ \sum_{k=1}^K \sigma^k (1 - \sum_{p \in P_k} \lambda^{k,p}) \\
= & \min_{\sigma^k, \pi_j \geq 0} ~ \max_{\lambda^{k,p} \geq 0} ~
\pi^\top b^0 + \sum_{k=1}^K \sigma^k
+ \sum_{k=1}^K \sum_{p \in P_k} \lambda^{k,p} \big( (c^k)^\top x^{k,p} - \pi^\top D^k \, x^{k,p} - \sigma^k \big)
\end{align}

which finally yields the dual problem to the full master problem $(\text{M})$

\begin{align}
\min_{\pi, \sigma \geq 0} ~& \pi^\top b^0 + \sum_{k=1}^K \sigma^k \\
\text{s.t.} ~ & \pi^\top D^k \, x^{k,p} + \sigma^k \geq (c^k)^\top x^{k,p} \quad \forall k, \forall p \in P_k.
\end{align}

Recall from the previous lecture the optimality conditions for a linear program: (1) primal feasibility, (2) dual feasibility, and (3) complementary slackness.
Now that we have optimized our restricted master problem, primal feasibility of the *full* master problem is guaranteed.
It is easy to check that complementary slackness also holds.

We are thus left to check whether dual feasibility holds.

The issue is that we cannot check the dual constraints one by one since we do not know the full set of extreme points $P_k$ for all $k$.
The key observation is that we actually do not need to know this set in order to determine whether at least one constraint is violated: we just need to study the following linear program (known as the reduced-cost problem).

**Proposition (Column generation).**
Let $(\pi, \sigma^k)$ be a dual solution to the optimization of the restricted master problem. 
For all $k$, the dual constraint $\pi^\top D^k \, x^{k,p} + \sigma^k \geq (c^k)^\top x^{k,p}$ is satisfied for all $p \in P_k$ if and only if the value of the linear program
\begin{align}
\max_{x^k \geq 0} ~ & \big( (c^k)^\top - \pi^\top D^k \big) x^k - \sigma^k \\
\text{s.t.} ~ & A^k x^k \leq b^k
\end{align}
is nonpositive.

If for some $k$, the program above takes a strictly positive value, it means that its solution is a new extreme point $x^{k,p}$ that we have to consider.
We thus add it to our restricted master problem together with its associated variable symbol $\lambda^{k,p}$, `self.lambda_vars[t,self.p_k[k]]`.
Again, we do not numerically solve for $\lambda^{k,p}$ yet.


In [18]:
def DantzigWolfe_column_generation(self, tol=1e-6):
    pi = np.array(self.master.getAttr("Pi", self.linking_j))
    sigma = np.array(self.master.getAttr("Pi", self.convexity_k))
    
    new_x_k = [ False for _ in range(self.K) ]
    for k in range(self.K):
        c_pricing = self.c_k[k] - self.D_k[k].T @ pi
        m = grb.Model()
        m.Params.OutputFlag = 0
        x = m.addMVar(self.G_k[k], lb=0.0)
        m.setObjective(c_pricing @ x, grb.GRB.MAXIMIZE)
        m.addConstr(self.A_k[k] @ x <= self.b_k[k])
        m.optimize()
        if m.Status == grb.GRB.OPTIMAL and m.ObjVal - sigma[k] > tol:
            self.p_k[k] += 1
            self.x_k_p[k].append(m.X)
            new_x_k[k] = m.X
    
    return new_x_k

DantzigWolfe.column_generation = DantzigWolfe_column_generation

## Full algorithm

We now summarize the steps of the algorithm:

---

**Dantzig—Wolfe algorithm.**

*Step 0.* For each $k$, compute an initial extreme point $x^{k,1} \in \arg\max_{x^k \geq 0} \big\{ (c^k)^\top x^k \; | \; A^k x^k \leq b^k \big\}$.  
Initialize the restricted master problem with the corresponding column $x^{k,1}$, and denote $\tilde P_k$ the set of generated columns up to now: $\tilde P_k = \{x^{k,1}\}$.

*Step 1.* Solve the restricted master problem
\begin{align*}
\max_{\lambda^{k,p} \geq 0} ~ & \sum_{k=1}^K \sum_{p \in \tilde P_k} \lambda^{k,p} \big( (c^k)^\top x^{k,p} \big) \\
\text{s.t.} ~ & \sum_{k=1}^K \sum_{p\in \tilde P_k} \lambda^{k,p} (D^k x^{k,p}) \leq b^0 \\
& \sum_{p \in \tilde P_k} \lambda^{k,p} \leq 1 \qquad (\forall k),
\end{align*}
and let $(\pi,\sigma^k)$ denote an optimal dual solution.

*Step 2.* For each $k$, solve the column generation problem
\begin{align*}
\max_{x^k \geq 0} ~ & \big( (c^k)^\top - \pi^\top D^k \big)x^k - \sigma^k \\
\text{s.t.} ~ & A^k x^k \leq b^k.
\end{align*}
If all problems have value $\leq 0$, stop. Otherwise, add every improving solution $x^k$ to corresponding set of generated columns $\tilde P_k$, and return to Step 1.

---

### Economic interpretation  
The dual variables $\pi_j$ are the shadow prices of the common resources.
Given these prices, in Step 2 factory $k$ chooses the feasible plan with the highest profit net of the social cost of the common resources it uses.

If this value is positive, factory $k$ has found a plan that raises total welfare, so the corresponding column should be added to the master problem.  
If no factory can do so, the current allocation is optimal.

Dantzig—Wolfe is thus **decentralization at work**: the master problem sets scarcity prices, and factories optimize locally at those prices.



We finally code the full algorithm with `DantzigWolfe_DWsolve`.


In [19]:
def DantzigWolfe_DWsolve(self, max_iter=100, tol=1e-6, verbose=1):
    self.initialize_master()
    new_x_k = self.initalize_extreme_points()
    it = 0

    while any(new_x_k) and it <= max_iter:
        it += 1
        self.update_master(new_x_k)
        self.master.optimize()
        new_x_k = self.column_generation()
        if verbose >= 1:
            print(f"Iter {it}: obj = {self.master.ObjVal:.6f}  (+{self.K - new_x_k.count(False)} columns)")

    if verbose >= 1 and it <= max_iter:
        print(f"Optimal after {it} iterations. Obj = {self.master.ObjVal:.6f}")
    
    x = np.hstack([
        sum(self.lambda_vars[k, p].X * np.array(self.x_k_p[k][p]) for p in range(self.p_k[k]+1))
        for k in range(self.K)
    ])
    
    return self.master.ObjVal, x

DantzigWolfe.DWsolve = DantzigWolfe_DWsolve

In [20]:
ex.DWsolve()

Iter 1: obj = 1652.403846  (+5 columns)
Iter 2: obj = 2502.690099  (+4 columns)
Iter 3: obj = 2590.388104  (+2 columns)
Iter 4: obj = 2593.279533  (+0 columns)
Optimal after 4 iterations. Obj = 2593.279533


(2593.2795329670334,
 array([50.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        16.25      , 14.375     ,  0.        , 34.87980769,  0.        ,
        13.64354396,  0.        , 28.8907967 ,  0.        , 32.85714286,
         0.        ,  0.        , 21.42857143]))

# Back to the assignment problem

Recall the large-scale assignment problem from our first lecture:
- Women $i$ are sorted into types $x \in X$, and men $j$ into types $y \in Y$, with type indicators $\delta_{ix}$ and $\delta_{jy}$.
- A match between woman $i$ of type $x$ and man $j$ of type $y$ creates surplus $\Phi_{xy} + \varepsilon_{iy} + \eta_{xj}$.
- The individual heterogeneity terms follow conditional laws $\varepsilon_{iy} \, | \, x_i = x \; \sim \; \mathbf P_x$ and $\eta_{xj} \, | \, y_j = y \; \sim \; \mathbf Q_y$.

The problem then writes:

\begin{align}
\max_{\pi_{iy}, \pi_{xj} \geq 0} ~ & \sum_i \bigg[ \pi_{i0} \varepsilon_{i0} + \sum_{y \in Y} \pi_{iy} \alpha_{iy} \bigg]
+ \sum_j \bigg[ \pi_{0j} \eta_{0j} + \sum_{x \in X} \pi_{xj} \gamma_{xj} \bigg] \\
\text{s.t.} ~ & \textstyle\sum_i \delta_{ix} \pi_{iy} = \textstyle\sum_j \delta_{jy} \pi_{xj} \qquad [w_{xy}] \\
              & \pi_{i0} + \textstyle\sum_{y \in Y} \pi_{iy} = 1 \quad\qquad [u_i] \\
              & \pi_{0j} + \textstyle\sum_{x \in X} \pi_{xj} = 1 \quad\qquad [v_j].
\end{align}
where
\begin{equation}
\alpha_{iy} = \sum_{x \in X} \delta_{ix} \frac{\Phi_{xy}}{2} + \varepsilon_{iy},
\qquad
\gamma_{xj} = \sum_{y \in Y} \delta_{jy} \frac{\Phi_{xy}}{2} + \eta_{xj}.
\end{equation}

This problem fits the Dantzig—Wolfe structure seen in our second lecture:
- Each individual feasibility constraint corresponds to a subproblem per woman $i$ and man $j$ ($I + J$ subproblems).
- The balance conditions $\textstyle\sum_i \delta_{ix} \pi_{iy} = \textstyle\sum_j \delta_{jy} \pi_{xj}$ are the linking constraints.

In this case, Dantzig–Wolfe turns out to have a very simple interpretation in we initially restrict individuals to be single, but gradually expand their choice set $Y_i$ or $X_j$ with new partner types.

---

**Algorithm: Dantzig–Wolfe for the assignment problem**

*Step 0.* Initialize the choice sets as $Y_i^0 = \emptyset$ for $i \in I$ and $X_j^0 = \emptyset$ for $j \in J$. Go to step 1.

*Step t.*  
1. Solve the restricted assignment problem using only variables in the current choice sets $Y_i = Y_i^t$ and $X_j = X_j^t$. Obtain a restricted optimal matching $\pi^t$ and its vectors of Lagrange multipliers $u^t, v^t, w^t$.
2. If
\begin{equation*}
u_i^t \geq \max_{y \in Y \setminus Y_i} ~ \alpha_{iy} - \sum_x \delta_{ix} w_{xy}^t \quad (\forall i)
\quad \text{and} \quad
v_j^t \geq \max_{x \in X \setminus X_j} ~ \gamma_{xj} + \sum_y \delta_{jy} w_{xy}^t \quad (\forall j),
\end{equation*}
stop.
Otherwise, go to 3.  
3. For each $i \in I$, if $i$ does not satisfy her above inequality, pick a type $y_i$ in the argmax and define $Y_i^{t+1} = Y_i^t \cup \{y_i\}$; otherwise, define $Y_i^{t+1} = Y_i^t$.
Similarly, for each $j \in J$, if $j$ does not satisfy his above inequality, pick a type $x_j$ in the argmax and define $X_j^{t+1} = X_j^t \cup \{x_j\}$; otherwise, define $X_j^{t+1} = X_j^t$.
Go to step $t+1$.

---

In [33]:
#!pip install --upgrade mec
import mec.dw

I, J = 8000, 6000
X, Y = 40, 30

np.random.seed(0)

x_i = np.random.randint(0, X, size=I)
y_j = np.random.randint(0, Y, size=J)

delta_i_x = np.eye(X)[x_i]
delta_j_y = np.eye(Y)[y_j]
assert (delta_i_x.sum(axis=0) > 0).all(), "Missing some types x"
assert (delta_j_y.sum(axis=0) > 0).all(), "Missing some types y"
print(f"Type assignments:")
print(f"Number of women by type: {delta_i_x.sum(axis=0)}")
print(f"Number of men by type:   {delta_j_y.sum(axis=0)}")

Phi_x_y = np.random.normal(0, 5, size=(X, Y))
eps_i_y = np.random.normal(0, 1, size=(I, Y)) / 10
eta_x_j = np.random.normal(0, 1, size=(X, J)) / 10
eps_i_0 = np.random.normal(0, 1, size=(I)) / 10
eta_0_j = np.random.normal(0, 1, size=(J)) / 10

alpha_i_y = delta_i_x @ Phi_x_y / 2 + eps_i_y
gamma_x_j = Phi_x_y @ delta_j_y.T / 2 + eta_x_j

ex = mec.dw.Dantzig_Wolfe(Phi_x_y, eps_i_y, eta_x_j, eps_i_0, eta_0_j, delta_i_x, delta_j_y)


# Solving the full master problem
_, _, total_time_full, _, model_full = ex.unrestricted_solve(verbose=0)

# Solving with Dantzig—Wolfe
_, total_time, _, _, _, model = ex.column_generation(max_iter=100, rc_tol=1e-6, verbose=0)


print('-----')

print('Time with Gurobi:', total_time_full)
print('Value:', model_full.ObjVal)

print('Time with Dantzig—Wolfe:', total_time)
print('Value:', model.ObjVal)


Type assignments:
Number of women by type: [215. 190. 179. 225. 192. 226. 180. 206. 194. 205. 188. 217. 203. 185.
 173. 185. 193. 230. 180. 178. 201. 199. 186. 191. 210. 203. 192. 220.
 206. 214. 212. 184. 191. 218. 192. 220. 201. 205. 219. 192.]
Number of men by type:   [189. 187. 203. 193. 188. 206. 211. 193. 214. 201. 213. 198. 199. 192.
 220. 206. 181. 197. 212. 189. 191. 201. 184. 204. 219. 213. 186. 197.
 211. 202.]

Total time: 5.2520804579835385
Iter 0: obj = 0.792717  (initial BFS)
Iter 13: no positive reduced cost – optimal.
-----
Time with Gurobi: 5.2520804579835385
Value: 60416.446703588146
Time with Dantzig—Wolfe: 1.3049245839938521
Value: 60416.44670358807
